In [2]:
!pip install pandas plotly ipywidgets

Defaulting to user installation because normal site-packages is not writeable


In [1]:
# =============================
# INSTALL IF NEEDED
# =============================
# !pip install pandas plotly ipywidgets numpy

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

# =============================
# CONFIG
# =============================
FILE_PATH = "powerbi_sales_dataset.csv"

# Region coordinate fallback for broad India regions
REGION_COORDS = {
    "North":   {"lat": 28.6139, "lon": 77.2090, "city": "Delhi"},
    "South":   {"lat": 12.9716, "lon": 77.5946, "city": "Bengaluru"},
    "East":    {"lat": 22.5726, "lon": 88.3639, "city": "Kolkata"},
    "West":    {"lat": 19.0760, "lon": 72.8777, "city": "Mumbai"},
    "Central": {"lat": 21.1458, "lon": 79.0882, "city": "Nagpur"}
}

REQUIRED_COLUMNS = [
    "ORDERDATE", "Region", "Category", "ProductName",
    "CustomerSegment", "SALES", "EstimatedProfit"
]

# =============================
# HELPERS
# =============================
def standardize_columns(df):
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", "", regex=True)
    )
    return df

def validate_columns(df):
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    return missing

def robust_parse_dates(series):
    s = series.astype(str).str.strip()

    parsed_1 = pd.to_datetime(s, format="%d-%b-%y", errors="coerce")
    parsed_2 = pd.to_datetime(s, format="%d-%b-%Y", errors="coerce")
    parsed_3 = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    parsed_4 = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    parsed_5 = pd.to_datetime(s, format="%m/%d/%Y", errors="coerce")
    parsed_6 = pd.to_datetime(s, errors="coerce", format="mixed")

    parsed = parsed_1.fillna(parsed_2).fillna(parsed_3).fillna(parsed_4).fillna(parsed_5).fillna(parsed_6)
    return parsed

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def add_geo_columns(df):
    if "lat" not in df.columns:
        df["lat"] = np.nan
    if "lon" not in df.columns:
        df["lon"] = np.nan

    if "Region" in df.columns:
        df["lat"] = df["lat"].fillna(df["Region"].map(lambda x: REGION_COORDS.get(x, {}).get("lat")))
        df["lon"] = df["lon"].fillna(df["Region"].map(lambda x: REGION_COORDS.get(x, {}).get("lon")))

    return df

def month_growth(series):
    if len(series) < 2:
        return 0.0
    prev_val = series.iloc[-2]
    curr_val = series.iloc[-1]
    if pd.isna(prev_val) or prev_val == 0:
        return 0.0
    return (curr_val - prev_val) / prev_val

def build_kpi_figure(total_revenue, total_profit, monthly_growth_pct):
    fig = go.Figure()

    fig.add_trace(go.Indicator(
        mode="number",
        value=total_revenue,
        number={"prefix": "₹", "valueformat": ",.0f"},
        title={"text": "Total Revenue"},
        domain={"x": [0.00, 0.30], "y": [0, 1]}
    ))

    fig.add_trace(go.Indicator(
        mode="number",
        value=total_profit,
        number={"prefix": "₹", "valueformat": ",.0f"},
        title={"text": "Total Profit"},
        domain={"x": [0.35, 0.65], "y": [0, 1]}
    ))

    fig.add_trace(go.Indicator(
        mode="number",
        value=monthly_growth_pct,
        number={"suffix": "%", "valueformat": ".2f"},
        title={"text": "Monthly Growth %"},
        domain={"x": [0.70, 1.00], "y": [0, 1]}
    ))

    fig.update_layout(
        title="Sales Dashboard KPIs",
        height=220,
        margin=dict(l=20, r=20, t=60, b=20),
        template="plotly_white"
    )
    return fig

def build_top_products_chart(dff):
    top_products = (
        dff.groupby("ProductName", as_index=False)["SALES"]
        .sum()
        .sort_values("SALES", ascending=False)
        .head(10)
    )

    fig = px.bar(
        top_products,
        x="SALES",
        y="ProductName",
        orientation="h",
        title="Top 10 Products by Revenue",
        color="SALES",
        color_continuous_scale="Blues"
    )
    fig.update_layout(
        yaxis={"categoryorder": "total ascending"},
        height=450,
        template="plotly_white"
    )
    return fig

def build_customer_segment_chart(dff):
    segment_sales = (
        dff.groupby("CustomerSegment", as_index=False)["SALES"]
        .sum()
        .sort_values("SALES", ascending=False)
    )

    fig = px.pie(
        segment_sales,
        names="CustomerSegment",
        values="SALES",
        hole=0.5,
        title="Customer Segments by Revenue"
    )
    fig.update_layout(height=450, template="plotly_white")
    return fig

def build_monthly_trend_chart(dff):
    monthly = (
        dff.groupby(dff["ORDERDATE"].dt.to_period("M"))["SALES"]
        .sum()
        .reset_index()
        .sort_values("ORDERDATE")
    )
    monthly["ORDERDATE"] = monthly["ORDERDATE"].astype(str)

    fig = px.line(
        monthly,
        x="ORDERDATE",
        y="SALES",
        markers=True,
        title="Monthly Revenue Trend"
    )
    fig.update_layout(height=450, template="plotly_white")
    return fig

def build_region_visual(dff):
    region_map = (
        dff.groupby(["Region", "lat", "lon"], as_index=False)["SALES"]
        .sum()
    ).dropna(subset=["lat", "lon"])

    if not region_map.empty:
        fig = px.scatter_geo(
            region_map,
            lat="lat",
            lon="lon",
            size="SALES",
            color="SALES",
            hover_name="Region",
            projection="natural earth",
            title="Sales by Region Map",
            color_continuous_scale="Viridis"
        )
        fig.update_layout(height=500, template="plotly_white")
        return fig

    region_sales = (
        dff.groupby("Region", as_index=False)["SALES"]
        .sum()
        .sort_values("SALES", ascending=False)
    )

    fig = px.bar(
        region_sales,
        x="Region",
        y="SALES",
        color="Region",
        title="Sales by Region"
    )
    fig.update_layout(height=450, template="plotly_white")
    return fig

# =============================
# LOAD + CLEAN DATA
# =============================
raw_df = pd.read_csv(FILE_PATH)
df = standardize_columns(raw_df.copy())

missing_cols = validate_columns(df)
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}. Available columns: {df.columns.tolist()}")

df["ORDERDATE"] = robust_parse_dates(df["ORDERDATE"])
df["SALES"] = safe_numeric(df["SALES"])
df["EstimatedProfit"] = safe_numeric(df["EstimatedProfit"])

for col in ["Region", "Category", "ProductName", "CustomerSegment"]:
    df[col] = df[col].astype(str).str.strip()

df = df.dropna(subset=["ORDERDATE", "SALES", "EstimatedProfit"]).copy()
df = add_geo_columns(df)

if df.empty:
    raise ValueError("No valid data left after cleaning. Check date and numeric columns in your CSV.")

# =============================
# WIDGETS
# =============================
min_date = df["ORDERDATE"].min().date()
max_date = df["ORDERDATE"].max().date()

start_date = widgets.DatePicker(description="Start Date", value=min_date)
end_date = widgets.DatePicker(description="End Date", value=max_date)

region_dropdown = widgets.Dropdown(
    options=["All"] + sorted(df["Region"].dropna().unique().tolist()),
    value="All",
    description="Region"
)

category_dropdown = widgets.Dropdown(
    options=["All"] + sorted(df["Category"].dropna().unique().tolist()),
    value="All",
    description="Category"
)

refresh_button = widgets.Button(
    description="Update Dashboard",
    button_style="primary",
    icon="refresh"
)

output = widgets.Output()

# =============================
# RENDER FUNCTION
# =============================
def render_dashboard(_=None):
    with output:
        clear_output(wait=True)

        dff = df.copy()

        if start_date.value is not None:
            dff = dff[dff["ORDERDATE"].dt.date >= start_date.value]

        if end_date.value is not None:
            dff = dff[dff["ORDERDATE"].dt.date <= end_date.value]

        if start_date.value is not None and end_date.value is not None:
            if start_date.value > end_date.value:
                print("Start Date cannot be greater than End Date.")
                return

        if region_dropdown.value != "All":
            dff = dff[dff["Region"] == region_dropdown.value]

        if category_dropdown.value != "All":
            dff = dff[dff["Category"] == category_dropdown.value]

        if dff.empty:
            print("No data available for the selected filters.")
            return

        monthly_sales = (
            dff.groupby(dff["ORDERDATE"].dt.to_period("M"))["SALES"]
            .sum()
            .sort_index()
        )

        total_revenue = dff["SALES"].sum()
        total_profit = dff["EstimatedProfit"].sum()
        monthly_growth_pct = month_growth(monthly_sales) * 100

        print("Dashboard Filters Applied")
        print(f"Rows: {len(dff):,}")
        print(f"Date Range: {dff['ORDERDATE'].min().date()} to {dff['ORDERDATE'].max().date()}")

        fig_kpi = build_kpi_figure(total_revenue, total_profit, monthly_growth_pct)
        fig_kpi.show()

        fig_top_products = build_top_products_chart(dff)
        fig_top_products.show()

        fig_segment = build_customer_segment_chart(dff)
        fig_segment.show()

        fig_trend = build_monthly_trend_chart(dff)
        fig_trend.show()

        fig_region = build_region_visual(dff)
        fig_region.show()

        preview_cols = [
            col for col in [
                "ORDERDATE", "OrderID", "CustomerID", "CustomerSegment",
                "Region", "Category", "ProductName", "SALES", "EstimatedProfit"
            ] if col in dff.columns
        ]

        display(dff[preview_cols].head(20))

refresh_button.on_click(render_dashboard)

controls = widgets.VBox([
    widgets.HTML("<h3>Sales Dashboard Filters</h3>"),
    start_date,
    end_date,
    region_dropdown,
    category_dropdown,
    refresh_button
])

display(controls, output)

render_dashboard()

Output()